# Practice 4 · Forward Propagation in CNNs and RNNs

**In class** · Goal: run your first code on HaiMa and learn how to interrupt it.

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |


## Question 1 · Single-Layer CNN Forward Propagation

Implement the forward pass of a single-layer convolutional neural network for a single-channel image.
The network contains three operations in order:
1. 2D valid convolution (no padding, stride = 1)
2. ReLU activation
3. 2×2 max pooling with stride = 2

Use only NumPy for computation.

**Function signature**:
`conv_pool_forward(x, kernel, bias)`

- `x`: input image, shape `(H, W)`
- `kernel`: convolution kernel, shape `(Kh, Kw)`
- `bias`: convolution bias scalar

**Return**: feature map after max pooling, shape `((H-Kh+1)//2, (W-Kw+1)//2)`

**Task**: 
Fill in the missing 6 lines of code for the convolution and pooling nested loops.


In [1]:
import numpy as np

def conv_pool_forward(x, kernel, bias):
    H, W = x.shape
    Kh, Kw = kernel.shape
    out_h_conv = H - Kh + 1
    out_w_conv = W - Kw + 1
    conv_out = np.zeros((out_h_conv, out_w_conv))
    
    # 🟢 EDIT: change this line (Loop over output spatial positions. Extract the local patch, compute element-wise multiplication with the kernel, sum them up, and add the bias)
    for i in range(out_h_conv):
        for j in range(out_w_conv):
            conv_out[i, j] = np.sum(x[i:i + Kh, j:j + Kw] * kernel) + bias

    relu_out = np.maximum(0, conv_out)
    pool_h = out_h_conv // 2
    pool_w = out_w_conv // 2
    pool_out = np.zeros((pool_h, pool_w))
    
    # 🟢 EDIT: change this line (Loop over pooling output positions. Extract the corresponding 2x2 region from the ReLU output and find its maximum value)
    for i in range(pool_h):
        for j in range(pool_w):
            pool_out[i, j] = np.max(relu_out[2*i:2*i + 2, 2*j:2*j + 2])

    return pool_out
    


## Question 2 · RNN Sequence Forward Propagation for Slot Filling

A vanilla RNN can process an input sequence word-by-word to output a probability distribution for each time step (e.g., Slot Filling). 
Implement the forward pass of a Many-to-Many RNN for a sequence using NumPy. Unlike a single step, you must process the entire sequence `X` and return the hidden states and output probabilities for **all** time steps.

**Formulas** (row-vector convention, consistent with the matrix shapes below; use NumPy `@` for matrix multiplication):
1. \( h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h) \)
2. \( y\_logits_t = h_t W_{hy} + b_y \)
3. \( y\_prob_t = \text{softmax}(y\_logits_t) \) (implement numerically stable softmax)

**Function signature**:
`rnn_sequence_forward(X, W_xh, W_hh, b_h, W_hy, b_y)`

- `X`: input sequence, shape `(seq_len, input_size)`
- `W_xh`: input-to-hidden weight, shape `(input_size, hidden_size)`
- `W_hh`: hidden-to-hidden weight, shape `(hidden_size, hidden_size)`
- `b_h`: hidden bias, shape `(hidden_size,)`
- `W_hy`: hidden-to-output weight, shape `(hidden_size, output_size)`
- `b_y`: output bias, shape `(output_size,)`

**Return**: `(H, Y_prob)`
- `H`: hidden states for all steps, shape `(seq_len, hidden_size)`
- `Y_prob`: output probabilities for all steps, shape `(seq_len, output_size)`


In [2]:
# Grid Type: Autograded answer
import numpy as np

def rnn_sequence_forward(X, W_xh, W_hh, b_h, W_hy, b_y):
    """Forward pass for a Many-to-Many RNN over an entire sequence."""
    seq_len, _ = X.shape
    hidden_size = W_hh.shape[0]
    output_size = W_hy.shape[1]
    
    H = np.zeros((seq_len, hidden_size))
    Y_prob = np.zeros((seq_len, output_size))
    
    # Initialize initial hidden state h_0 as zero vector
    h_prev = np.zeros(hidden_size)
    
    for t in range(seq_len):
        x_t = X[t]
        
        # 🟢 EDIT: change this line (Compute the hidden state, logits, stable softmax, and update the previous hidden state)
        h_t = np.tanh(x_t @ W_xh + h_prev @ W_hh + b_h)
        logits = h_t @ W_hy + b_y
        exp_logits = np.exp(logits - np.max(logits))
        H[t] = h_t
        Y_prob[t] = exp_logits / np.sum(exp_logits)
        h_prev = h_t
        
    return H, Y_prob


## Question 3 · Many-to-One RNN Binary Classification

Implement a many-to-one RNN model for binary sequence classification.
1. Process the full input sequence step by step with a vanilla RNN cell (tanh activation).
2. Take the **final hidden state** after processing the whole sequence.
3. Map the final hidden state to a scalar through a linear output layer.
4. Apply sigmoid to get the final classification probability.

**Function signature**:
`rnn_many_to_one(x_seq, W_xh, W_hh, b_h, W_out, b_out)`

- `x_seq`: input sequence, shape `(seq_len, input_size)`
- `W_xh`, `W_hh`, `b_h`: RNN cell parameters
- `W_out`: output weight matrix, shape `(hidden_size, 1)`
- `b_out`: output bias scalar

**Return**: final classification probability (scalar)


In [3]:
import numpy as np

def sigmoid(x):
    x = np.asarray(x, dtype=float)
    exp_neg_abs = np.exp(-np.abs(x))
    return np.where(x >= 0, 1.0 / (1.0 + exp_neg_abs),
                    exp_neg_abs / (1.0 + exp_neg_abs))

def rnn_many_to_one(x_seq, W_xh, W_hh, b_h, W_out, b_out):
    """
    Many-to-one RNN for binary classification.
    Parameters:
        x_seq: input sequence, shape (seq_len, input_size)
        W_xh: input-to-hidden weight, shape (input_size, hidden_size)
        W_hh: hidden-to-hidden weight, shape (hidden_size, hidden_size)
        b_h: hidden bias, shape (hidden_size,)
        W_out: output weight, shape (hidden_size, 1)
        b_out: output bias scalar
    Returns:
        y_prob: final classification probability, scalar
    """
    h = np.zeros(W_hh.shape[0])
    for x_t in x_seq:
        h = np.tanh(x_t @ W_xh + h @ W_hh + b_h)

    logit = (h @ W_out).item() + b_out
    return float(sigmoid(logit))
    